# 实验五 · 融合算子 —— 用减少访存量换取性能

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：40–50 分钟

前四个实验每次只实现一个算子。真实的神经网络不是这样运行的：算子一个接一个执行，每一个都要把数据从 Global Memory 读进来、算完再写回去。中间结果只是下一个算子的输入，若把相邻的算子合并成一个核函数、让中间结果留在片上，就可以省去这几趟往返。

这正是官方《算子开发指南》在矢量计算优化中列为**高优先级**的一项建议——「通过 Unified Buffer 融合实现连续 vector 计算」。本实验把这条建议做成可以测量的实验：沿一条三算子的链逐步提高融合程度，核算访存量的变化，再用实测检验它能兑现多少。

> **实验说明**
> 1. 本实验沿用递进式的版本组织：融合的程度逐版本提高，每一版只引入一个新概念。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 全部源码写入同一个 `.asc` 文件，由一条 `bisheng` 命令编译。
> 6. 本实验建立在**实验二**（三段流水与分块）与**实验四**（Sigmoid 的基础 API 分解）之上，建议先完成这两个实验。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明算子融合的收益来源，指出它减少的究竟是哪一部分开销
- 精确核算一条算子链在融合前后的 Global Memory 访存量
- 实现多算子链的三种融合程度，理解中间张量为何要占用额外的设备内存
- 说明同一个 Stream 上多次下发的串行语义，以及它与算子链先后关系的对应
- 掌握片上缓冲的就地复用方法，说明基础 API 与高阶 API 在地址重叠上的规定差异
- 判断一条算子链的节拍受限于搬运还是受限于计算，据此说明融合的收益边界
- 说明融合的代价：片上压力上升、缓冲占用时间变长、算子粒度变粗

## 🗺️ 学习路径

1. **准备阶段**：确定算子链，核算融合前后的访存量与片上占用
2. **v1 · 三算子独立**：三次核函数启动，两个中间张量落回 Global Memory
3. **v2 · 部分融合**：前两个算子合并，只剩一个中间张量
4. **v3 · 全融合**：一次启动，中间张量消失
5. **v4 · 就地复用**：把输出缓冲当作中间缓冲，取消核内的临时缓冲
6. **结果可视化与分析**：加速比与访存量之比的对照，以及融合收益边界的判断


## 1. 背景与动机：融合减少的是什么

### 1.1 本实验的算子链

本实验选取一条三算子的链，它是神经网络中常见的线性变换、激活、门控三步结构：

$$ t_1 = a x + b, \qquad t_2 = \sigma(t_1), \qquad y = t_2 \odot v $$

合起来即

$$ y_i = \sigma(a x_i + b) \cdot v_i $$

其中 $a = 0.5$、$b = -0.25$ 为标量，$x$ 与 $v$ 是两个长度 $N = 2^{21}$ 的 `float` 张量，$\odot$ 表示逐元素相乘。中间的 $\sigma$ 沿用实验四的四条基础指令分解，因此整条链一共 **7 条矢量指令**：

`Muls` → `Adds` → `Muls` → `Exp` → `Adds` → `Reciprocal` → `Mul`

### 1.2 融合不改变本实验这条链的计算量

先明确一点：**本实验的融合不改变任何一次浮点运算**。上面 7 条指令，无论分成三个核函数还是合成一个，条数完全相同。融合减少的只是中间结果 $t_1$、$t_2$ 在 Global Memory 上的往返。

<img src="./images/06.05_fusion_traffic.png" alt="06.05_fusion_traffic"  width="900px" >

官方文档为这一优化画出了数据流的前后对照。图中的反例把两次矢量计算拆成两轮，每一轮都要从 Global Memory 搬入、算完再搬回 Global Memory；正例只搬入一次，在 Unified Buffer 内连续算完再搬出一次：

<img src="./images/06.05_ub_fusion_dataflow.png" alt="06.05_ub_fusion_dataflow" width="800px">

*数据流图对比，通过 Unified Buffer 融合实现连续 vector 计算*

官方在该节给出了通式：需要进行的矢量计算为 $n$ 次时，不融合的实现从 Global Memory 搬进搬出共 $2n$ 次，而融合之后只需一次搬入、一次搬出。本实验的算子链正是这一通式的实例，第 1.3 节把它折算成字节数。

> **一处口径差异**：官方在介绍融合算子的总体优势时列有减少计算量一条，指的是把多个算子合并后可以省去重复的中间步骤。本实验刻意选了一条没有重复步骤的链，因此计算量严格不变——这样做是为了让访存量成为唯一的变量，便于把收益归因清楚。

### 1.3 访存量核算

设每个元素占 4 字节，逐个核函数累加读写的字节数：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 各核函数的访存 | 总访存 | 相对 v1 的比值 |
| --- | --- | --- | --- |
| v1 三算子独立 | $(4N{+}4N) + (4N{+}4N) + (4N{+}4N{+}4N)$ | $28N$ | 1.00 |
| v2 部分融合 | $(4N{+}4N) + (4N{+}4N{+}4N)$ | $20N$ | 1.40 |
| v3 全融合 | $4N{+}4N{+}4N$ | $12N$ | **2.33** |
| v4 全融合 + 就地复用 | $4N{+}4N{+}4N$ | $12N$ | 2.33 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">各核函数的访存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">总访存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">相对 v1 的比值</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 三算子独立</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">(4N{+}4N) + (4N{+}4N) + (4N{+}4N{+}4N)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">28N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1.00</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 部分融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">(4N{+}4N) + (4N{+}4N{+}4N)</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">20N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1.40</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4N{+}4N{+}4N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">12N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>2.33</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v4 全融合 + 就地复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4N{+}4N{+}4N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">12N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2.33</td>
</tr>
</tbody>
</table>

最后一列是**访存这一项**所允许的加速比上界。

读与写的字节数在这里直接相加，是有依据的。官方给出搬运流水的理论耗时算法：搬运数据量除以理论带宽；并且明确指出，当 MTE2 与 MTE3 同时读写 Global Memory 时，搬运流水线的耗时应按（MTE2 搬运量 + MTE3 搬运量）除以 Global Memory 带宽核算。搬入与搬出由两条流水分别发起，但它们共用同一条 Global Memory 通路，因此核算搬运耗时时字节数仍要相加。

**这个上界成立的前提是耗时与访存量成正比，也就是算子完全受搬运限制；这个前提在本实验中并不成立。**实验四第 13 节已经用实测指出过：`Exp` 一类超越函数的代价明显高于一次普通运算，扣除固定开销之后耗时并不与搬运量成正比。本实验会把这条线索接着往下追：矢量计算本身也占一条流水，融合把七条指令集中到一个核函数之后，它可能取代搬运成为最慢的一条。第 11 节用实测检验这一点。

## 2. 三处实现要点

### 2.1 中间张量要占用设备内存

$t_1$ 与 $t_2$ 既不是算子的输入也不是输出，而是链内部的暂存数据。它们必须在 Global Memory 上真实存在，因此 v1 需要额外申请 $2 \times 8$ MiB 设备内存，v2 需要 8 MiB，v3 与 v4 不需要。

**在推理场景中，设备内存往往是比耗时更硬的约束**：一个模型能否装进设备，可能就取决于中间张量是否被融合消除。

### 2.2 三次下发的先后由 Stream 保证

v1 的三个核函数下发到同一个 Stream 上。同一 Stream 内的任务按下发顺序串行执行，因此第二个核函数必然在第一个全部完成之后才开始，无须显式同步——这与实验三 v3 的两阶段规约依靠的是同一条性质。

代价也随之而来：串行意味着每一次都要重新承担启动开销，并重新填充与排空三段流水。

启动开销有多大？官方给出了一个直观的参照——空 Kernel 的耗时，即不含任何计算、只有启动过程的核函数所花的时间：

<img src="./images/06.05_head_overhead_vs_cores.png" alt="06.05_head_overhead_vs_cores" width="400px">

*头开销随启动核数的变化。这部分时延被称为头开销，其中包含核启动、核取址 TLB MISS、同地址访问以及变量资源初始化；并指出头开销会随着使用的核数增加而增加*

可见一次核函数启动本身就有微秒量级的固定代价。v1 要付三次，v3 只付一次。不过在本实验的数据规模下，这一项在总耗时中的占比不大——**融合的收益主体仍然来自访存量的减少**，第 11 节会回到这一点。

<img src="./images/06.05_pipeline_timeline.png" alt="06.05_pipeline_timeline"  width="880px" >

请留意这张图所依据的规则：**MTE2、Vector、MTE3 是三条可以并行工作的流水线，因此一个核函数每处理一个分块所花的时间，由三者中最慢的一条决定，而不是三者之和。**

这条规则说的是三条流水在时间上可以重叠，不等于三者互不影响。MTE2 与 MTE3 各自独立发起搬运，却共用同一条 Global Memory 通路，因此在核算搬运这一条流水的耗时时，读与写的字节数仍要相加（依据见第 1.3 节所引官方口径）。把它写成一句可以直接使用的核算规则：一个核函数处理一个分块的时间，约为**搬运总量所需的时间**与**矢量指令所需的时间**两者中较大的一个。第 11 节用这条规则解释实测的结果。

### 2.3 源与目的地址重叠

基础算术 API 的接口约束中没有禁止源与目的地址重叠，因此 `Adds(t, t, 1, n)` 这样的就地写法是允许的，本实验的四个版本都用到了它。v4 不再另设中间缓冲，全部七条指令直接在输出缓冲上完成。

**高阶 API 的规定不同**：`AscendC::Sigmoid` 一类高阶接口内部要自行申请临时空间、分多步完成计算，其接口约束明确规定：**不支持源操作数与目的操作数地址重叠，也不支持 `sharedTmpBuffer` 与源操作数、目的操作数地址重叠**。这就是本实验的 v4 只使用基础算术 API 的原因——高阶 `Sigmoid` 无法参与就地复用。使用任何高阶接口做就地复用之前，都应当先查阅《Ascend C API 参考》中该接口的约束说明。

## 3. 片上缓冲的占用核算

分块长度取 `TILE_LENGTH = 4096` 个 `float`，即每块 16 KB；队列深度取 2，因此每个 `TQue` 占两块。按实验二确立的做法，在写代码之前先把占用算清楚：

<img src="./images/06.05_ub_budget.png" alt="06.05_ub_budget"  width="880px" >

三点值得注意：

- **融合把片上压力集中到了一个核函数里。** v1 的三个核函数各自占 80 KB 或 96 KB，v3 单个核函数即占 112 KB。
- **v4 与 v3 的访存量完全相同**，差别只在片上少占了 16 KB。因此不应指望 v4 在耗时上有明显改善——它省下的是空间，不是时间。
- 省下的空间可以换成更大的分块长度。`TILE_LENGTH` 翻倍到 8192 时，v3 需要 224 KB，超出 UB 容量；v4 需要 192 KB，恰好等于 UB 的标称容量。按实验二第 12 节的口径，这一档属于边界情况：核算公式给出的是可用上限而非安全取值，能否编译运行还取决于编译器为临时变量留出多少空间。动手练习第 1 题按这个口径实测。

分块长度还有另一重考虑。官方指出，单次搬运的数据量过小时达不到理论带宽，根据实测经验，单次搬运长度在 16 KB 以上通常能较好地发挥带宽性能：

<img src="./images/06.05_bw_vs_blocksize_gm2ub.png" alt="06.05_bw_vs_blocksize_gm2ub" width="400px">
<img src="./images/06.05_bw_vs_blocksize_ub2gm.png" alt="06.05_bw_vs_blocksize_ub2gm" width="400px">

*GM->UB 方向、UB->GM 方向不同单次搬运数据量下实际占用带宽的变化。图中数据与处理器型号相关，实测存在抖动，仅用于说明趋势*

本实验取 `TILE_LENGTH = 4096` 个 `float`，单次搬运恰为 16 KB，正落在这条曲线由陡转平的位置。这也是这个取值的依据：再小会掉进带宽利用率不足的一段，测到的耗时就不再只反映访存量的差别。


## 4. 本实验的测量方法

### 4.1 两个口径与容差

本实验只用两个耗时口径：CPU 基准 `cpu_ms` 与核函数耗时 `kernel_ms`；预热、重复取平均、同步之后再停表这三条计时纪律与实验二至实验四一致，此处不再复述。

**不设端到端口径**，理由与前几个实验相同，在本实验中还要更强一些：真实推理中一条算子链的输入搬上设备之后会连续参与多个算子，直到整段计算图算完才回传，逐个算子测量一次完整往返并不对应任何实际的用法；何况本实验四个版本的主机与设备之间传输量完全相同，把这一项计入只会把版本间的差异按同一个常数摊薄。融合与卸载的关系放在第 11 节 ② 讨论。

校验采用相对误差判定，容差取 `5e-3`。取值依据与实验四相同：链上有 `Reciprocal`，这条硬件近似求倒数接口的相对误差在 $10^{-3}$ 量级，是整条链上精度最低的一环。详细推导见实验四第 4.2 节。
### 4.2 CPU 基准也做两个版本

本实验的 CPU 基准有两份实现：**一趟算完**（融合）与**分三趟算**（不融合）。后者不参与加速比，只用于回答一个问题：融合是昇腾平台特有的手段，还是一条不依赖具体硬件的原理？

CPU 侧的三趟实现同样要把 $t_1$、$t_2$ 写出去再读回来，机制与 NPU 上的 v1 完全对应。至于这一项在 CPU 上到底值多少时间，取决于中间数组能否留在末级缓存里，以及 CPU 侧的瓶颈本身是不是访存——第 11 节 ⑤ 给出实测答案。

加速比一律以**一趟算完的 CPU 版本**为基准，即拿 NPU 与 CPU 上更好的那份实现相比，以免高估收益。



## 5. 环境准备与检查

In [ ]:
!mkdir -p src_fusion

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[
            :1800
        ]
    )

## 6. 版本设计总览

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 新增的唯一概念 | 启动次数 | 中间张量 | 核内临时缓冲 | 片上占用 | 访存量 |
| --- | --- | --- | --- | --- | --- | --- |
| **v1** | 算子链的独立实现与中间张量 | 3 | 2 个（16 MiB） | 有 | 80 / 80 / 96 KB | $28N$ |
| **v2** | 部分融合 | 2 | 1 个（8 MiB） | 有 | 80 / 96 KB | $20N$ |
| **v3** | 全融合 | **1** | **0 个** | 有 | **112 KB** | $12N$ |
| **v4** | 就地复用输出缓冲 | 1 | 0 个 | **无** | 96 KB | $12N$ |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">新增的唯一概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">启动次数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">中间张量</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核内临时缓冲</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">片上占用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">访存量</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子链的独立实现与中间张量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2 个（16 MiB）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">80 / 80 / 96 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">28N</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">部分融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 个（8 MiB）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">80 / 96 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">20N</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>0 个</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>112 KB</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">12N</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">就地复用输出缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0 个</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>无</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">96 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">12N</td>
</tr>
</tbody>
</table>

一条线索贯穿四个版本：**v1 → v3 减少的是 Global Memory 访存量，v3 → v4 减少的是片上缓冲占用**。前者影响耗时，后者影响可选的分块长度上界。

代码上的对应关系同样清晰。七条指令按融合程度分给不同的核函数，但三段流水的骨架一次都没有变过：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 核函数 | 输入个数 | 承担的指令 | 用于哪些版本 |
| --- | --- | --- | --- |
| `KernelUnary<OP_LINEAR>` | 1 | `Muls` `Adds` | v1 ① |
| `KernelUnary<OP_SIGMOID>` | 1 | `Muls` `Exp` `Adds` `Reciprocal` | v1 ② |
| `KernelUnary<OP_LIN_SIG>` | 1 | 前六条 | v2 ① |
| `KernelBinary<OP_MUL>` | 2 | `Mul` | v1 ③、v2 ② |
| `KernelBinary<OP_FUSED>` | 2 | 全部七条，用临时缓冲 | v3 |
| `KernelBinary<OP_FUSED_IP>` | 2 | 全部七条，就地完成 | v4 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核函数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">输入个数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">承担的指令</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用于哪些版本</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelUnary&lt;OP_LINEAR&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Muls</code> <code>Adds</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ①</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelUnary&lt;OP_SIGMOID&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Muls</code> <code>Exp</code> <code>Adds</code> <code>Reciprocal</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ②</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelUnary&lt;OP_LIN_SIG&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">前六条</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 ①</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelBinary&lt;OP_MUL&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Mul</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ③、v2 ②</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelBinary&lt;OP_FUSED&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全部七条，用临时缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelBinary&lt;OP_FUSED_IP&gt;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全部七条，就地完成</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v4</td>
</tr>
</tbody>
</table>

因此只需要写**两个类**：一个一元、一个二元。模板实参 `OP` 是编译期常量，`Compute()` 里的分支在编译期即被折叠，不会留下运行期判断。

## 7. Device 侧实现

### 7.1 文件头与参数

先写入头文件与全部可调参数。与前三个实验一样，参数一律用 `#ifndef` 包裹，以便在动手练习中用 `-D` 在命令行上覆盖，无须改动源码。

In [ ]:
%%writefile src_fusion/ascendc_fusion.asc
/**
 * 并行计算 第六章 实验五：融合算子
 *
 * 本文件包含六个核函数、CPU 基准、数据生成、校验与 main，
 * 由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_fusion/ascendc_fusion.asc --npu-arch=dav-2201 -O2 -lm -o src_fusion/ascendc_fusion
 *
 * 与实验四相同，Host 侧用到 std::exp，因此 -lm 不能省略。
 *
 * 用法：
 *   ./ascendc_fusion         四个版本对照，N 取默认值
 *   ./ascendc_fusion <N>     指定元素总数
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>  // 计时：clock_gettime
#include <vector>

#include "acl/acl.h"          // Host 侧
#include "kernel_operator.h"  // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

/* 元素总数的默认值：2^21 = 2 097 152，每个 float 张量 8 MiB */
#ifndef LAB_TOTAL_LENGTH
#define LAB_TOTAL_LENGTH (2 * 1024 * 1024)
#endif
constexpr uint32_t TOTAL_LENGTH = static_cast<uint32_t>(LAB_TOTAL_LENGTH);

/* 分块长度：单位是元素而非字节，4096 个 float 即每块 16 KB */
#ifndef LAB_TILE_LENGTH
#define LAB_TILE_LENGTH (4 * 1024)
#endif
constexpr uint32_t TILE_LENGTH = static_cast<uint32_t>(LAB_TILE_LENGTH);

/* 参与计算的核数 */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数：预热次数与重复次数 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (50)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 3;

/* 队列深度：TQue 模板的第二个参数，编译期常量 */
constexpr uint32_t QUEUE_DEPTH = 2;

/* 算子链的两个标量系数：y = sigma(a * x + b) * v */
constexpr float COEF_A = 0.5f;
constexpr float COEF_B = -0.25f;

/* 相对误差容差。链上有 Reciprocal——硬件近似求倒数接口，相对误差在 1e-3 量级，
 * 是整条链上精度最低的一环，容差由它决定。推导见实验四 4.2 节 */
constexpr double RTOL = 5e-3;

/* Compute() 里要执行哪几条指令，由模板实参选定。这些值是编译期常量，
 * 因此 Compute() 中的分支在编译期即被折叠，不会留下运行期判断 */
constexpr int32_t OP_LINEAR = 1;  /* d = a*x + b                    v1 ①    */
constexpr int32_t OP_SIGMOID = 2; /* d = sigma(x)                   v1 ②    */
constexpr int32_t OP_LIN_SIG = 3; /* d = sigma(a*x + b)             v2 ①    */
constexpr int32_t OP_MUL = 11; /* d = x * v                      v1③ v2② */
constexpr int32_t OP_FUSED = 12; /* d = sigma(a*x + b) * v，用临时缓冲  v3 */
constexpr int32_t OP_FUSED_IP = 13; /* 同上，全部就地完成                v4 */

### 7.2 一元核函数 `KernelUnary<OP>`

一个输入、一个输出，覆盖 v1 的前两个核函数与 v2 的第一个核函数。

三段流水的骨架、核间切分与核内分块，与实验二逐行相同，**唯一随 `OP` 变化的是 `Compute()` 里的指令序列**。请留意其中的就地写法：`Adds(t, t, COEF_B, ...)` 的源与目的是同一块缓冲，这在基础算术 API 上是允许的。

还有一点需要说明：本类为所有 `OP` 都分配了一块临时缓冲 `tmpBuf`，但就 `OP_LINEAR` 与 `OP_SIGMOID` 本身而言，这块缓冲并不是必需的——两者的指令序列都可以完全在输出缓冲上就地完成，例如线性变换写成 `Muls(d, s, COEF_A, ...); Adds(d, d, COEF_B, ...);` 即可省去 16 KB。这里保留它，是为了让 v1、v2、v3 在片上缓冲的用法上保持一致，把取消临时缓冲这一步单独留给 v4 演示，见第 7.3 节。

In [ ]:
%%writefile -a src_fusion/ascendc_fusion.asc
/* ============ 一元逐元素算子：一个输入、一个输出 ============
 * 新增概念：把算子链的一段封装成核函数，链的划分由模板实参决定
 */
template <int32_t OP>
class KernelUnary {
 public:
  __aicore__ inline KernelUnary() {}

  __aicore__ inline void Init(GM_ADDR src, GM_ADDR dst, uint32_t n) {
    /* 核间切分：与实验二、三、四相同，由 blockIdx 算出本核负责的区间 */
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    srcGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(src) + offset,
                          blockLength_);
    dstGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(dst) + offset,
                          blockLength_);

    pipe.InitBuffer(inQueue, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(tmpBuf, TILE_LENGTH * sizeof(float));
  }

  __aicore__ inline void Process() {
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<float> s = inQueue.AllocTensor<float>();
    AscendC::DataCopy(s, srcGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueue.EnQue(s);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<float> s = inQueue.DeQue<float>();
    AscendC::LocalTensor<float> d = outQueue.AllocTensor<float>();
    AscendC::LocalTensor<float> t = tmpBuf.Get<float>();

    if (OP == OP_LINEAR) { /* 两条指令：d = a*x + b */
      AscendC::Muls(t, s, COEF_A, TILE_LENGTH);
      AscendC::Adds(d, t, COEF_B, TILE_LENGTH);
    } else {
      if (OP == OP_LIN_SIG) { /* 先做线性变换，再取负 */
        AscendC::Muls(t, s, COEF_A, TILE_LENGTH);
        AscendC::Adds(t, t, COEF_B, TILE_LENGTH);
        AscendC::Muls(t, t, -1.0f, TILE_LENGTH);
      } else { /* OP_SIGMOID：输入即 t1，直接取负 */
        AscendC::Muls(t, s, -1.0f, TILE_LENGTH);
      }
      /* 以下三条与实验四 v1 的后三条逐条相同 */
      AscendC::Exp(t, t, TILE_LENGTH);
      AscendC::Adds(t, t, 1.0f, TILE_LENGTH);
      AscendC::Reciprocal(d, t, TILE_LENGTH);
    }

    outQueue.EnQue(d);
    inQueue.FreeTensor(s);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<float> d = outQueue.DeQue<float>();
    AscendC::DataCopy(dstGm[progress * TILE_LENGTH], d, TILE_LENGTH);
    outQueue.FreeTensor(d);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::GlobalTensor<float> srcGm, dstGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 7.3 二元核函数 `KernelBinary<OP>`

两个输入、一个输出，覆盖 v1 与 v2 的最后一个核函数，以及 v3、v4 这两个全融合版本。

与一元版本相比只多了一处：`CopyIn` 要搬入两块数据。**注意 $v$ 的搬入时机**——它只在最后一条 `Mul` 中被读取，却必须与 $x$ 在同一个分块循环内搬入，因为两者要在元素上一一对应。代价是 `inQueueV` 在整个 `Compute` 期间都被占用，这是融合的一项固有开销。

`OP_FUSED` 与 `OP_FUSED_IP` 的差别只有一处：前者把中间量写在 `tmpBuf` 上，最后一条指令是 `Mul(d, t, v)`；后者把中间量直接写在输出缓冲 `d` 上，最后一条是 `Mul(d, d, v)`。**把 `t` 换成 `d`，`tmpBuf` 就不再需要**——这就是 v4 省下 16 KB 的全部原因。

In [ ]:
%%writefile -a src_fusion/ascendc_fusion.asc
/* ============ 二元逐元素算子：两个输入、一个输出 ============
 * 新增概念：融合——把整条链放进一个 Compute()，中间量不再离开片上
 */
template <int32_t OP>
class KernelBinary {
 public:
  __aicore__ inline KernelBinary() {}

  __aicore__ inline void Init(GM_ADDR src, GM_ADDR gate, GM_ADDR dst,
                              uint32_t n) {
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(src) + offset,
                        blockLength_);
    vGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(gate) + offset,
                        blockLength_);
    dstGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(dst) + offset,
                          blockLength_);

    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(inQueueV, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    pipe.InitBuffer(outQueue, QUEUE_DEPTH, TILE_LENGTH * sizeof(float));
    /* 只有 OP_FUSED 需要核内临时缓冲：OP_MUL 只有一条指令，
     * OP_FUSED_IP 把输出缓冲当作中间缓冲用，二者都不需要额外的 16 KB */
    if (OP == OP_FUSED) {
      pipe.InitBuffer(tmpBuf, TILE_LENGTH * sizeof(float));
    }
  }

  __aicore__ inline void Process() {
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
      CopyOut(i);
    }
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<float> x = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> v = inQueueV.AllocTensor<float>();
    AscendC::DataCopy(x, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    AscendC::DataCopy(v, vGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(x);
    inQueueV.EnQue(v);
  }

  __aicore__ inline void Compute() {
    AscendC::LocalTensor<float> x = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> v = inQueueV.DeQue<float>();
    AscendC::LocalTensor<float> d = outQueue.AllocTensor<float>();

    if (OP == OP_MUL) { /* 链的最后一步单独成核 */
      AscendC::Mul(d, x, v, TILE_LENGTH);
    } else if (OP == OP_FUSED) { /* v3：七条指令，中间量放临时缓冲 */
      AscendC::LocalTensor<float> t = tmpBuf.Get<float>();
      AscendC::Muls(t, x, COEF_A, TILE_LENGTH);
      AscendC::Adds(t, t, COEF_B, TILE_LENGTH);
      AscendC::Muls(t, t, -1.0f, TILE_LENGTH);
      AscendC::Exp(t, t, TILE_LENGTH);
      AscendC::Adds(t, t, 1.0f, TILE_LENGTH);
      AscendC::Reciprocal(t, t, TILE_LENGTH);
      AscendC::Mul(d, t, v, TILE_LENGTH);
    } else { /* v4：同样七条指令，把 t 换成 d，临时缓冲随之消失 */
      AscendC::Muls(d, x, COEF_A, TILE_LENGTH);
      AscendC::Adds(d, d, COEF_B, TILE_LENGTH);
      AscendC::Muls(d, d, -1.0f, TILE_LENGTH);
      AscendC::Exp(d, d, TILE_LENGTH);
      AscendC::Adds(d, d, 1.0f, TILE_LENGTH);
      AscendC::Reciprocal(d, d, TILE_LENGTH);
      AscendC::Mul(d, d, v, TILE_LENGTH);
    }

    outQueue.EnQue(d);
    inQueueX.FreeTensor(x);
    inQueueV.FreeTensor(v);
  }

  __aicore__ inline void CopyOut(uint32_t progress) {
    AscendC::LocalTensor<float> d = outQueue.DeQue<float>();
    AscendC::DataCopy(dstGm[progress * TILE_LENGTH], d, TILE_LENGTH);
    outQueue.FreeTensor(d);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueV;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::GlobalTensor<float> xGm, vGm, dstGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 7.4 六个核函数入口

四个版本一共用到六个核函数入口。每一个入口只有三行：声明任务类型、实例化对应的模板、调用 `Init` 与 `Process`。

In [ ]:
%%writefile -a src_fusion/ascendc_fusion.asc
/* ===================== 核函数入口 ===================== */

extern "C" __global__ __aicore__ void fusion_linear(GM_ADDR x, GM_ADDR t1,
                                                    uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); /* 声明为纯矢量内核 */
  KernelUnary<OP_LINEAR> op;
  op.Init(x, t1, n);
  op.Process();
}

extern "C" __global__ __aicore__ void fusion_sigmoid(GM_ADDR t1, GM_ADDR t2,
                                                     uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelUnary<OP_SIGMOID> op;
  op.Init(t1, t2, n);
  op.Process();
}

extern "C" __global__ __aicore__ void fusion_lin_sig(GM_ADDR x, GM_ADDR t,
                                                     uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelUnary<OP_LIN_SIG> op; /* 与上面两个的唯一差别：模板实参 */
  op.Init(x, t, n);
  op.Process();
}

extern "C" __global__ __aicore__ void fusion_mul(GM_ADDR t, GM_ADDR v,
                                                 GM_ADDR y, uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelBinary<OP_MUL> op;
  op.Init(t, v, y, n);
  op.Process();
}

extern "C" __global__ __aicore__ void fusion_all(GM_ADDR x, GM_ADDR v,
                                                 GM_ADDR y, uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelBinary<OP_FUSED> op;
  op.Init(x, v, y, n);
  op.Process();
}

extern "C" __global__ __aicore__ void fusion_all_inplace(GM_ADDR x, GM_ADDR v,
                                                         GM_ADDR y,
                                                         uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelBinary<OP_FUSED_IP> op; /* 与 fusion_all 的唯一差别：模板实参 */
  op.Init(x, v, y, n);
  op.Process();
}

## 8. Host 侧实现

### 8.1 工具函数

ACL 返回值检查、计时、数据生成与校验的写法与前三个实验一致，此处不再复述。本实验只有两处是新的：

- **CPU 基准写了两份**：`RunOnCpuFused` 一趟算完，`RunOnCpuThreePass` 分三趟并把 $t_1$、$t_2$ 写回内存。后者不参与加速比，用于第 11 节的一项对照。
- **校验采用相对误差判定**。本算子的输出含因子 $v$，取值可以任意接近 0，用绝对误差判定会失去分辨力。分母按《编写规范》取 $\max(10^{-6}, |golden|)$。

In [ ]:
%%writefile -a src_fusion/ascendc_fusion.asc
/* ============================================================
 *                       Host 侧代码
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

static void NpuInit() {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));
}

static void NpuFinalize() {
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());
}

/* ---------- 数据生成：线性同余发生器，输出 [-1, 1] ---------- */
static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u;
  return static_cast<float>(state >> 8) * (2.0f / 16777216.0f) - 1.0f;
}

static void GenerateInput(std::vector<float> &a, uint32_t seed) {
  uint32_t s = seed;
  for (size_t i = 0; i < a.size(); ++i) {
    a[i] = LcgNextFloat(s);
  }
}

/* ---------- 参考真值：以 double 计算整条链 ---------- */
static inline double ChainRef(double xv, double vv) {
  return vv / (1.0 + std::exp(-(COEF_A * xv + COEF_B)));
}

/* ---------- CPU 基准一：一趟算完（融合） ----------
 * 结果写入实际数组并参与误差统计，因此不会被编译器消除 */
static double RunOnCpuFused(const std::vector<float> &x,
                            const std::vector<float> &v, std::vector<float> &y,
                            int warmup, int repeat) {
  const size_t n = x.size();
  for (int r = 0; r < warmup; ++r) {
    for (size_t i = 0; i < n; ++i) {
      y[i] = v[i] / (1.0f + std::exp(-(COEF_A * x[i] + COEF_B)));
    }
  }
  const double t0 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) {
    for (size_t i = 0; i < n; ++i) {
      y[i] = v[i] / (1.0f + std::exp(-(COEF_A * x[i] + COEF_B)));
    }
  }
  return (GetTimeMs() - t0) / repeat;
}

/* ---------- CPU 基准二：分三趟算，中间结果写回内存 ----------
 * 与 NPU 的 v1 结构一致：每一趟都要完整遍历一次数组。
 * N = 2^21 时每个数组 8 MiB，远超末级缓存，因此 t1、t2 一定会落到内存 */
static double RunOnCpuThreePass(const std::vector<float> &x,
                                const std::vector<float> &v,
                                std::vector<float> &t1, std::vector<float> &t2,
                                std::vector<float> &y, int warmup, int repeat) {
  const size_t n = x.size();
  for (int r = 0; r < warmup; ++r) {
    for (size_t i = 0; i < n; ++i) t1[i] = COEF_A * x[i] + COEF_B;
    for (size_t i = 0; i < n; ++i) t2[i] = 1.0f / (1.0f + std::exp(-t1[i]));
    for (size_t i = 0; i < n; ++i) y[i] = t2[i] * v[i];
  }
  const double t0 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) {
    for (size_t i = 0; i < n; ++i) t1[i] = COEF_A * x[i] + COEF_B;
    for (size_t i = 0; i < n; ++i) t2[i] = 1.0f / (1.0f + std::exp(-t1[i]));
    for (size_t i = 0; i < n; ++i) y[i] = t2[i] * v[i];
  }
  return (GetTimeMs() - t0) / repeat;
}

/* ---------- 校验：相对误差判定，绝对误差一并报告 ---------- */
static bool Verify(const char *ver, uint32_t n, const std::vector<float> &out,
                   const std::vector<double> &golden, double rtol) {
  double maxAbs = 0.0, maxRel = 0.0;
  for (size_t i = 0; i < golden.size(); ++i) {
    const double g = golden[i];
    const double a = std::fabs(static_cast<double>(out[i]) - g);
    if (a > maxAbs) maxAbs = a;
    /* 输出含因子 v，可以任意接近 0，分母按《编写规范》做下界保护 */
    const double denom = std::fabs(g) > 1e-6 ? std::fabs(g) : 1e-6;
    const double r = a / denom;
    if (r > maxRel) maxRel = r;
  }
  const bool ok = (maxRel <= rtol);
  std::printf(
      "[VERIFY] ver=%s n=%u rtol=%.1e max_abs_err=%.3e max_rel_err=%.3e "
      "result=%s\n",
      ver, n, rtol, maxAbs, maxRel, ok ? "PASS" : "FAIL");
  return ok;
}

/* ---------- 打印一行可被程序解析的性能记录 ----------
 * launches   本版本的核函数启动次数
 * traffic    每个元素引起的 Global Memory 读写字节数（28 / 20 / 12）
 * ub_kb      本版本各核函数中片上占用最大的那一个
 * ws_mib     中间张量占用的设备内存 */
static void ReportPerf(const char *ver, uint32_t n, uint32_t launches,
                       uint32_t traffic, double ubKB, double wsMiB,
                       double cpuMs, double kernelMs) {
  std::printf(
      "[PERF]   ver=%s n=%u blockDim=%u launches=%u traffic=%u ub_kb=%.1f "
      "ws_mib=%.1f cpu_ms=%.4f kernel_ms=%.4f sp_kernel=%.4f\n",
      ver, n, BLOCK_DIM, launches, traffic, ubKB, wsMiB, cpuMs, kernelMs,
      cpuMs / kernelMs);
}

/* ---------- 计时宏：纯核函数耗时（不含 H2D/D2H） ---------- */
#define TIME_KERNEL(LAUNCH, OUT_MS)                                               \
  do {                                                                            \
    for (int _w = 0; _w < WARMUP; ++_w) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream));                                \
    }                                                                             \
    const double _t0 = GetTimeMs();                                               \
    for (int _r = 0; _r < REPEAT; ++_r) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream)); /* 先同步再停止计时 */ \
    }                                                                             \
    (OUT_MS) = (GetTimeMs() - _t0) / REPEAT;                                      \
  } while (0)

### 8.2 主程序

`main` 的结构与前三个实验一致：造数 → CPU 基准 → 申请显存 → 逐版本运行、校验、计时。

有两处是本实验特有的。

第一，**四个版本的下发序列各封装成一个函数**。`LaunchV1` 依次下发三个核函数，它们在同一个 `g_stream` 上，因此按下发顺序串行执行——这正是第 2.2 节所说的 Stream 语义。封装的目的是让 `TIME_KERNEL` 能够整体重复整个下发序列，而不必为每个版本各写一遍计时循环。

第二，**中间张量 `t1Dev`、`t2Dev` 一次性申请**。v3 与 v4 完全不使用它们；真实部署中这 16 MiB 根本不会被申请，此处一次申请只是为了让四个版本共用一套代码。

In [ ]:
%%writefile -a src_fusion/ascendc_fusion.asc
/* ---------- 四个版本的下发序列 ----------
 * 同一个 g_stream 上的任务按下发顺序串行执行，
 * 因此 v1 的三个核函数无须任何显式同步即可保证先后关系。 */
static void LaunchV1(uint8_t *x, uint8_t *v, uint8_t *t1, uint8_t *t2,
                     uint8_t *y, uint32_t n) {
  fusion_linear<<<BLOCK_DIM, nullptr, g_stream>>>(x, t1, n);
  fusion_sigmoid<<<BLOCK_DIM, nullptr, g_stream>>>(t1, t2, n);
  fusion_mul<<<BLOCK_DIM, nullptr, g_stream>>>(t2, v, y, n);
}

static void LaunchV2(uint8_t *x, uint8_t *v, uint8_t *t1, uint8_t *y,
                     uint32_t n) {
  fusion_lin_sig<<<BLOCK_DIM, nullptr, g_stream>>>(x, t1, n);
  fusion_mul<<<BLOCK_DIM, nullptr, g_stream>>>(t1, v, y, n);
}

static void LaunchV3(uint8_t *x, uint8_t *v, uint8_t *y, uint32_t n) {
  fusion_all<<<BLOCK_DIM, nullptr, g_stream>>>(x, v, y, n);
}

static void LaunchV4(uint8_t *x, uint8_t *v, uint8_t *y, uint32_t n) {
  fusion_all_inplace<<<BLOCK_DIM, nullptr, g_stream>>>(x, v, y, n);
}

/* 每个版本的固定流程：运行一次并校验 -> 计时 -> 报告。
 * 输入在四个版本之间保持常驻设备，主机与设备之间的传输不计入耗时：
 * 判断融合收益的单位是整段计算图，而不是单个算子的一次往返，理由见 4.1 节 */
#define RUN_VERSION(TAG, LAUNCH, LAUNCHES, TRAFFIC, UBKB, WSMIB)    \
  do {                                                              \
    LAUNCH;                                                         \
    ACL_CHECK(aclrtSynchronizeStream(g_stream));                    \
    ACL_CHECK(aclrtMemcpy(yHost.data(), bytes, yDev, bytes,         \
                          ACL_MEMCPY_DEVICE_TO_HOST));              \
    allPass &= Verify(TAG, n, yHost, golden, RTOL);                 \
    TIME_KERNEL(LAUNCH, kMs);                                       \
    ReportPerf(TAG, n, LAUNCHES, TRAFFIC, UBKB, WSMIB, cpuMs, kMs); \
  } while (0)

int32_t main(int32_t argc, char *argv[]) {
  uint32_t n = TOTAL_LENGTH;
  if (argc > 1) n = static_cast<uint32_t>(std::strtoul(argv[1], nullptr, 10));

  const uint32_t granularity = BLOCK_DIM * TILE_LENGTH;
  if (n == 0 || n % granularity != 0) {
    std::printf(
        "[FATAL] N 必须是 BLOCK_DIM x TILE_LENGTH = %u 的整数倍，当前 N = %u\n",
        granularity, n);
    return 1;
  }

  /* ---------- 数据准备：两个输入各用一个种子，真值以 double 计算 ---------- */
  std::vector<float> x(n), v(n), yHost(n), cpuOut(n), cpuT1(n), cpuT2(n);
  std::vector<double> golden(n);
  GenerateInput(x, 2026u);
  GenerateInput(v, 20260521u);
  for (uint32_t i = 0; i < n; ++i) {
    golden[i] = ChainRef(static_cast<double>(x[i]), static_cast<double>(v[i]));
  }

  /* ---------- 两份 CPU 基准；加速比一律以一趟算完的那份为准 ---------- */
  const int cpuRepeat = (REPEAT <= 1) ? 1 : 5;
  const int cpuWarmup = (REPEAT <= 1) ? 1 : WARMUP;
  const double cpuMs = RunOnCpuFused(x, v, cpuOut, cpuWarmup, cpuRepeat);
  const double cpuThreeMs =
      RunOnCpuThreePass(x, v, cpuT1, cpuT2, yHost, cpuWarmup, cpuRepeat);
  double cpuRel = 0.0;
  for (uint32_t i = 0; i < n; ++i) {
    const double g = golden[i];
    const double denom = std::fabs(g) > 1e-6 ? std::fabs(g) : 1e-6;
    const double r = std::fabs(static_cast<double>(cpuOut[i]) - g) / denom;
    if (r > cpuRel) cpuRel = r;
  }

  std::printf("N=%u  TILE_LENGTH=%u  BLOCK_DIM=%u  REPEAT=%d  a=%.3f  b=%.3f\n",
              n, TILE_LENGTH, BLOCK_DIM, REPEAT, COEF_A, COEF_B);
  std::printf(
      "[BASE]   n=%u cpu_ms=%.4f cpu_three_pass_ms=%.4f cpu_max_rel_err=%.3e\n",
      n, cpuMs, cpuThreeMs, cpuRel);

  /* ---------- 申请显存 ---------- */
  NpuInit();
  const size_t bytes = static_cast<size_t>(n) * sizeof(float);
  uint8_t *xDev = nullptr, *vDev = nullptr, *yDev = nullptr;
  uint8_t *t1Dev = nullptr, *t2Dev = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&xDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&vDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&yDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  /* 中间张量：只有 v1 与 v2 使用。v1 用两块，v2 只用 t1Dev 一块 */
  ACL_CHECK(aclrtMalloc((void **)&t1Dev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&t2Dev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(
      aclrtMemcpy(xDev, bytes, x.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(
      aclrtMemcpy(vDev, bytes, v.data(), bytes, ACL_MEMCPY_HOST_TO_DEVICE));

  bool allPass = true;
  double kMs = 0.0;

  /* 参数依次为：标签、下发序列、启动次数、每元素访存字节、片上占用、中间张量。
   * 下发序列写成函数调用，全部逗号都在圆括号内，可以安全地作为宏参数传递 */
  RUN_VERSION("v1", LaunchV1(xDev, vDev, t1Dev, t2Dev, yDev, n), 3, 28, 96.0,
              16.0);
  RUN_VERSION("v2", LaunchV2(xDev, vDev, t1Dev, yDev, n), 2, 20, 96.0, 8.0);
  RUN_VERSION("v3", LaunchV3(xDev, vDev, yDev, n), 1, 12, 112.0, 0.0);
  RUN_VERSION("v4", LaunchV4(xDev, vDev, yDev, n), 1, 12, 96.0, 0.0);

  /* ---------- 释放资源（与申请严格成对，顺序相反） ---------- */
  ACL_CHECK(aclrtFree(t2Dev));
  ACL_CHECK(aclrtFree(t1Dev));
  ACL_CHECK(aclrtFree(yDev));
  ACL_CHECK(aclrtFree(vDev));
  ACL_CHECK(aclrtFree(xDev));
  NpuFinalize();

  std::printf(allPass ? "[SUCCESS] 全部版本通过校验。\n"
                      : "[FAILED] 存在未通过校验的版本！\n");
  return allPass ? 0 : 1;
}

## 9. 编译与运行

`-O2` 不能省略：CPU 基准与 NPU 代码在同一次编译中生成，用 `-O0` 会人为放大基准实现的耗时，使加速比失真。`-lm` 同样不能省略，原因与实验四相同——Host 侧用到 `std::exp`，而 `bisheng` 默认不链接数学库。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_fusion/ascendc_fusion.asc"
EXE = "src_fusion/ascendc_fusion"

# 编译选项集中定义一处，§12 的参数扫描直接复用，避免两处不一致
FLAGS = ["--npu-arch=" + ARCH, "-O2", "-lm"]

# bisheng [算子源文件] [编译选项] -o [输出产物名称]
cmd = ["bisheng", SRC] + FLAGS + ["-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

一次运行即可输出四个版本的校验结果与性能记录。

In [ ]:
def run_demo(args=(), exe=None, timeout=900):
    # 与实验二、三、四同名同约定：只返回 stdout；exe 缺省为 §9 编译出的主程序
    proc = subprocess.run(
        ["./" + (exe or EXE)] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print(out_main)

### 9.1 解析输出

三类记录行的字段名固定，按空格切分即可解析为字典。

In [ ]:
def parse_rows(text, tag):
    # 把所有以 [tag] 开头的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[" + tag + "]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k in ("ver", "result") else float(v)
            rows.append(d)
    return rows


def parse_perf(text):
    return parse_rows(text, "PERF")


def parse_verify(text):
    return {r["ver"]: r for r in parse_rows(text, "VERIFY")}


def parse_base(text):
    rows = parse_rows(text, "BASE")
    return rows[0] if rows else None


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
base_main = parse_base(out_main)
base_v1 = rows_main[0]["kernel_ms"] if rows_main else 1.0

print(
    "CPU 基准（一趟算完）：%.4f ms     CPU 分三趟：%.4f ms     三趟/一趟 = %.2fx"
    % (
        base_main["cpu_ms"],
        base_main["cpu_three_pass_ms"],
        base_main["cpu_three_pass_ms"] / base_main["cpu_ms"],
    )
)
print("CPU float32 相对 double 真值的最大相对误差：%.3e" % base_main["cpu_max_rel_err"])
print()

hdr = (
    "版本",
    "启动",
    "访存/元素",
    "中间张量",
    "UB(KB)",
    "CPU(ms)",
    "kernel(ms)",
    "vs CPU",
    "vs v1",
    "最大相对误差",
    "判定",
)
print("%-5s %5s %10s %10s %8s %9s %11s %9s %8s %13s %6s" % hdr)
print("-" * 112)
for r in rows_main:
    v = chk_main[r["ver"]]
    print(
        "%-5s %5d %9dN %8.0f MiB %8.1f %9.4f %11.4f %8.2fx %7.2fx %13.3e %6s"
        % (
            r["ver"],
            r["launches"],
            r["traffic"],
            r["ws_mib"],
            r["ub_kb"],
            r["cpu_ms"],
            r["kernel_ms"],
            r["sp_kernel"],
            base_v1 / r["kernel_ms"],
            v["max_rel_err"],
            v["result"],
        )
    )

## 10. 结果可视化

**第一张图**是四个版本相对 CPU 基准的加速比。口径只计核函数耗时；主机与设备之间的传输不参与比较，理由见第 4.1 节。四个版本执行的浮点运算完全相同，因此柱高的差别只反映数据流经 Global Memory 的次数。

**第二张图**把版本之间的实测加速比与第 1.3 节算出的访存量之比画在一起。若耗时确实与访存量成正比，两组柱应当等高；差额为负，说明存在搬运之外的因素在拖慢。这个差额是第 11 节的主题。


In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_KERNEL, C_E2E, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
spk = [r["sp_kernel"] for r in rows_main]
xpos = np.arange(len(vers))

fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=120)
ax.bar(xpos, spk, 0.46, color=C_KERNEL, label="NPU kernel time")
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.text(
    -0.45, 1.06, "baseline: CPU 1 thread, fused = 1.0x", fontsize=9, color="#777777"
)
for i, a in enumerate(spk):
    ax.text(i, a, "%.1fx" % a, ha="center", va="bottom", fontsize=8.5)
ax.set_xticks(xpos)
ax.set_xticklabels(
    [
        "%s\n(%d launch, %dN)" % (r["ver"], r["launches"], r["traffic"])
        for r in rows_main
    ]
)
ax.set_yscale("log")
ax.set_ylabel("Speedup over CPU baseline (kernel time)")
ax.set_title("Lab 5: fusion - NPU kernel speedup vs single-thread CPU")
ax.grid(axis="y", alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
ratio_meas = [base_v1 / r["kernel_ms"] for r in rows_main]
ratio_theo = [rows_main[0]["traffic"] / r["traffic"] for r in rows_main]
xpos = np.arange(len(vers))

fig, ax = plt.subplots(figsize=(8.0, 4.4), dpi=120)
ax.bar(xpos - 0.19, ratio_theo, 0.38, color=C_BASE, label="traffic ratio (theory)")
ax.bar(xpos + 0.19, ratio_meas, 0.38, color=C_OPT, label="measured kernel speedup")
for i, (a, b) in enumerate(zip(ratio_theo, ratio_meas)):
    ax.text(i - 0.19, a, "%.2fx" % a, ha="center", va="bottom", fontsize=8.5)
    ax.text(i + 0.19, b, "%.2fx" % b, ha="center", va="bottom", fontsize=8.5)
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.set_xticks(xpos)
ax.set_xticklabels(["%s\n%dN" % (r["ver"], r["traffic"]) for r in rows_main])
ax.set_ylabel("Ratio vs. v1")
ax.set_title("Lab 5: measured speedup vs. Global Memory traffic ratio")
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

print("版本   访存量之比   实测加速比   差额")
for r, a, b in zip(rows_main, ratio_theo, ratio_meas):
    print("%-5s %10.2fx %11.2fx %8.2fx" % (r["ver"], a, b, b - a))

## 11. 结果分析

> 本节讨论的是从上面的输出中应当读出什么，以及这些结论在多大范围内成立。具体数值请以各自机器上的实际运行结果为准：不同型号的机器数值会有差别。如果这台服务器同时还有其他用户在使用，实测耗时会有明显波动，个别数值甚至可能出现反常（例如核数增加而耗时不降）——这属于机器负载带来的干扰，判断时应看整体走向，不要在单个数值上做解释。

**① 实测的收益低于访存量之比，因为计算也占一条流水**

上一节的第二张图里，实测的版本间加速比明显低于访存量之比。先想清楚这个差额意味着什么：**访存量之比只有在耗时完全由搬运决定时才可能达到；实测低于它，就说明至少有一个版本的耗时并不是由搬运决定的。**

按第 2.2 节的规则，一个核函数处理一个分块的时间，由搬运总量与矢量指令两者中较慢的一个决定。把各个核函数的这两项数出来：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 核函数 | 每个分块搬运的数组个数 | 矢量指令条数 |
| --- | --- | --- |
| v1 ① 线性变换 | 2（读 $x$、写 $t_1$） | 2 |
| v1 ② Sigmoid | 2（读 $t_1$、写 $t_2$） | 4 |
| v1 ③ 逐元素乘 | 3（读 $t_2$、读 $v$、写 $y$） | 1 |
| v2 ① 线性变换 + Sigmoid | 2（读 $x$、写 $t_1$） | 6 |
| v3 / v4 全融合 | 3（读 $x$、读 $v$、写 $y$） | 7 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核函数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">每个分块搬运的数组个数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">矢量指令条数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ① 线性变换</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2（读 $x$、写 $t_1$）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ② Sigmoid</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2（读 $t_1$、写 $t_2$）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 ③ 逐元素乘</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3（读 $t_2$、读 $v$、写 $y$）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 ① 线性变换 + Sigmoid</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2（读 $x$、写 $t_1$）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">6</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 / v4 全融合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3（读 $x$、读 $v$、写 $y$）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">7</td>
</tr>
</tbody>
</table>

对照这张表：v1 ③ 只有一条指令却要搬三个数组，节拍显然由搬运决定；v1 ① 两者相当，也偏向搬运。**指令与搬运之比最高的是全融合版本——七条指令只对应三个数组的搬运，而这七条里还包含 `Exp` 与 `Reciprocal` 两条代价明显高于加减乘的超越函数。**

也就是说，融合在减少搬运的同时，把矢量计算的负担从分摊在三个核函数变成了集中在一个核函数。当矢量计算越过搬运成为最慢的一条流水时，**再减少搬运也不会更快**。

这就是**融合的收益边界**。本实验的算子链恰好走到了这条边界上，这正是实测低于访存量之比的主要原因。

还有一部分差额来自核函数启动：v1 要付三次启动开销，v3 只付一次，两者相差不大（量级见第 2.2 节的官方图）。要把「差额中有多少来自计算、有多少来自启动」定量地分开，需要用 Profiling 观察各条流水的实际耗时与空 Kernel 的启动代价，属于进阶内容，本实验不展开。**本实验能够确证的是：收益的主体来自访存量的减少，而它受一条明确的边界约束。**

**② 融合与卸载：判断的单位是整段计算图**

本实验没有测量「搬上设备—计算—搬回主机」这一整套流程的耗时，这不是遗漏。

融合改变的是设备内部的数据流动，对主机与设备之间的往返没有影响；本实验四个版本的传输量又完全相同。把这一项计入，只会给四个版本加上同一个常数，把它们的差异按同一个比例摊薄，得到的比值主要反映这台机器上主机与设备之间的通路有多快，与融合本身无关——不同机器上这个比值可以相差一个数量级，因而不具备可比性。

更重要的是它不对应真实的用法。真实推理中，输入搬上设备之后会连续参与许多算子，直到整段计算图算完才回传一次。**判断一次卸载是否划算，单位应当是整段计算图，而不是其中的某一个算子。** 若某段计算图卸载到设备上确实不划算，正确的应对是把更多算子留在设备上、减少往返次数，而不是在单个算子上继续做融合。

**③ v3 与 v4 的耗时差异应先与运行间波动比较**

两者的访存量与指令序列完全相同，差别只在 v3 多用了 16 KB 片上缓冲。因此**不应预期两者的耗时有可分辨的差别**。实测也确实如此：两者之差与同一版本在多次运行之间的差处于同一量级，方向甚至可能在不同机器上相反。**这种量级的差异不足以支持任何机制上的解释，如实记录即可。**

v4 的收益是释放出的 16 KB 片上空间，它要通过调整分块长度才能兑现，见动手练习第 1 题。**把优化等同于耗时下降，是一种常见的误解。**

**④ 四个版本的最大相对误差完全相同**

四个版本、不同机器，`max_rel_err` 一位不差地相同，且与实验四中同样使用 `Reciprocal` 的版本一致。同一条 `Reciprocal`，在不同的算子、不同的输入分布、不同的融合程度下给出同一个相对误差——这是链上精度最低的一环决定整体精度最直接的证据。

它同时说明**融合本身不引入额外的精度损失**：七条指令的执行顺序没有变，变的只是中间量存放在哪里，而中间量在片上与在 Global Memory 上都是同一种 `float`。

由此也可以核对 `RTOL = 5e-3` 的取值是否合适：实测误差与容差之间应当留有余量，但余量不宜过大，否则容差就失去了判别能力。若把 `Reciprocal` 换成 `Div`（实验四动手练习第 4 题），这条容差可以显著收紧。

**⑤ CPU 上分三趟同样慢于一趟算完，但幅度因机器而异**

融合在 CPU 上同样有收益，方向与 NPU 一致——这说明它不是昇腾平台特有的手段，而是一条与硬件无关的原理。但幅度在不同主机上差别很大，远不如 NPU 侧的比值稳定，原因有两条：

- **CPU 侧的瓶颈未必是访存。** 一趟算完的每元素耗时由 `expf` 主导，省下的那一趟搬运摊在一个本来就被超越函数占满的循环上，占比自然小。主机的单核性能越强，这一项占比越小。
- **中间数组未必真的落到了内存。** $t_1$、$t_2$ 各 8 MiB，在末级缓存较大的服务器 CPU 上可能大部分仍是缓存命中，因此多出来的两趟并没有付出内存往返的全价，而末级缓存的容量因机器而异。

**因此这一项只能得到方向上的结论，得不到幅度上的结论。** 想把第二条原因单独检验出来，可以按动手练习第 6 题把中间数组放大到明显超过本机末级缓存。

这两条恰好又回到本实验的主线：融合的收益上界，由被省掉的那部分搬运在总节拍中所占的比重决定。无论在 NPU 还是在 CPU 上，只要计算成了瓶颈，这个比重就小。

**⑥ 融合消除的不只是访存**

v1 需要 16 MiB 设备内存存放两个中间张量，v2 需要 8 MiB，v3 与 v4 不需要。在推理场景中，设备内存往往是比耗时更硬的约束：一个模型能否装进设备，可能就取决于中间张量是否被融合消除。**这一项收益不会出现在任何一条耗时曲线上，却可能是融合最重要的理由。**

**⑦ 融合的代价**

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 代价 | 表现 |
| --- | --- |
| 片上压力上升 | 全融合版本占用的 Unified Buffer 高于 v1 的任何一个核函数，可选的分块长度上界因此下降 |
| 缓冲占用时间变长 | $v$ 只在最后一条指令被读取，却在整个 <code>Compute</code> 期间占着一块输入缓冲 |
| <strong>计算集中，可能越过收益边界</strong> | 见 ①：七条指令挤进一个核函数后，矢量流水成为最慢的一条 |
| 算子粒度变粗 | 融合算子专用于这一条链，换一条链就要重写 |
| 可选实现受限 | 高阶 API 不支持源与目的地址重叠，无法参与就地复用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">代价</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">表现</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上压力上升</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全融合版本占用的 Unified Buffer 高于 v1 的任何一个核函数，可选的分块长度上界因此下降</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">缓冲占用时间变长</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$v$ 只在最后一条指令被读取，却在整个 <code>Compute</code> 期间占着一块输入缓冲</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>计算集中，可能越过收益边界</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">见 ①：七条指令挤进一个核函数后，矢量流水成为最慢的一条</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子粒度变粗</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合算子专用于这一条链，换一条链就要重写</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可选实现受限</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API 不支持源与目的地址重叠，无法参与就地复用</td>
</tr>
</tbody>
</table>

---

### 🎓 结论

融合揭示了一条与前几个实验不同的优化路径：**前面各版本改进的是计算的组织方式，融合改进的是数据流经 Global Memory 的次数。**

但本实验的收获不止于融合更快这一条，更在于知道这条路走到哪里为止：

1. 按官方口径核算搬运量——读与写共用同一条 Global Memory 通路，字节数相加——得到一个上界；
2. 这个上界只在耗时完全由搬运决定时才可能达到。矢量计算也占一条流水，而融合把七条指令集中到了一个核函数，它因此成了最慢的一条；
3. 于是实测必然低于上界，**继续减少访存也不再有效**。

优化的下一步不是接着融合，而是换一个方向：或者让计算本身更省，或者把整段计算图留在设备上、减少主机与设备之间的往返。


## 12. 🔧 动手练习

1. **验证 v4 释放的空间确实能兑现。** 按第 3 节的核算，`TILE_LENGTH` 取 8192 时 v3 需要 224 KB、超出 UB 容量，v4 需要 192 KB、恰好等于标称容量。后者属于边界情况：实验二第 12 节已经指出，核算公式给出的是可用上限而非安全取值，能否跑通还取决于编译器为临时变量留出多少空间。用 `-DLAB_TILE_LENGTH=8192` 重新编译运行，先记录失败发生在编译阶段还是运行阶段；再把 `KernelBinary` 中 `OP_FUSED` 分支改成与 `OP_FUSED_IP` 一致（即去掉 `tmpBuf`），看 192 KB 这一档在本机上能否通过，并如实记录结果。注意四个版本编译在同一个可执行文件里，只要有一个版本超限，整个程序都无法产出结果。
2. **把 v3 拆成两个核函数，验证收益边界。** 第 11 节 ① 指出 v3 的节拍已经由矢量计算决定。据此预测：若把 v3 的七条指令拆成前四条与后三条两个核函数（中间量落回 Global Memory，访存量升到 $20N$），耗时**不一定**变差。动手实现并测量，看预测是否成立。
3. **推迟 $v$ 的搬入。** 把 `KernelBinary` 的 `CopyIn` 拆成两半，$v$ 改到前六条指令之后再搬入。正确性是否受影响？耗时是否变化？请结合缓冲的占用时间解释。
4. **延长算子链。** 在链末再加一步 $y' = y + c$，分别实现四算子独立与全融合两个版本。先按第 11 节 ① 的口径判断：全融合版本的矢量指令增加到八条之后，它的瓶颈仍在计算一侧吗？再测量核对。
5. **换用高阶 API。** 把 v3 的四条 sigmoid 指令替换为 `AscendC::Sigmoid`。代码短了多少？性能有何变化？再依据第 2.3 节给出的接口约束说明：它为什么不能照 v4 的写法做就地复用？
6. **让 CPU 的中间数组真正落到内存。** 第 11 节 ⑤ 指出 $t_1$、$t_2$ 各 8 MiB 时很可能仍是缓存命中。用 `-DLAB_TOTAL_LENGTH` 把 $N$ 加大到中间数组明显超过本机末级缓存（先用 `lscpu` 查一下），同时用 `-DLAB_REPEAT=5` 缩短耗时，重测三趟与一趟之比，看它是否上升。
7. **改变核数。** 用 `-DLAB_BLOCK_DIM` 依次取 1、2、4、8、16、32 重新编译运行，记录 v1 与 v3 的核函数耗时。两者是否都随核数成比例下降？在核数很多、单核计算量很小时，为什么下降会变慢？请结合第 2.2 节的官方头开销图解释。


## 13. 🤔 思考题

- 融合不改变任何一次浮点运算，那么它究竟减少了什么？**请用字节而非指令作答。**
- 第 1.3 节按官方口径把读与写的字节数相加，理由是 MTE2 与 MTE3 共用同一条 Global Memory 通路。那么在什么情况下，把搬入与搬出分别计入两条流水、取二者的较大值，会是更接近实际的近似？
- 第 11 节 ① 指出，全融合版本的瓶颈已经在矢量计算一侧。**如果把数据类型从 `float` 换成 `half`，搬运所需的时间与矢量计算所需的时间各会怎么变？** 融合的收益是变大还是变小？（提示：参考实验四关于 `Exp` 处理同样数量的 `half` 与 `float` 耗时的结论。）
- 假设某条链的中间张量远小于输入输出（例如中间是一个标量）。此时融合的收益会如何变化？
- v3 的 `inQueueV` 被占用了整个 `Compute` 期间，而 $v$ 只在最后一条指令被读取。这一浪费能否消除？如果可以，代价是什么？
- 实验四的 v2 使用了高阶 `Sigmoid`，本实验的 v4 只用基础算术 API。请说明理由，并讨论：高阶 API 是否就此不适合用于融合算子？在什么情况下它仍然是更好的选择？
- 融合在 CPU 与 NPU 上兑现的幅度相差很大，而且 CPU 侧的幅度还随主机而变。两者所说的把中间结果留在近处，分别指留在哪里？为什么同一条原理在两种硬件上兑现的幅度差别这么大？
- 融合算子的可复用性低于独立算子。在什么情况下这一代价是可以接受的？这与编译器所做的算子融合优化是什么关系？


## 14. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 融合的收益来源 | 不改变本实验这条链的计算量，减少中间结果在 Global Memory 上的往返；官方将其列为高优先级优化项「通过 Unified Buffer 融合实现连续 vector 计算」 |
| 访存量核算 | 逐核函数累加读写字节数；本链为 $28N \to 20N \to 12N$ |
| <strong>读与写的字节数应当相加</strong> | MTE2 与 MTE3 各自独立发起搬运，却共用同一条 Global Memory 通路。官方给出：两者同时读写时，搬运流水的耗时按（MTE2 搬运量 + MTE3 搬运量）÷ Global Memory 带宽核算 |
| <strong>节拍由最慢的一条流水决定</strong> | 搬运与计算可以在时间上重叠；一个分块的时间取搬运总量所需的时间与矢量指令所需的时间之中较大的一个 |
| <strong>访存量之比是宽松的上界</strong> | 它假定耗时与访存量成正比，忽略了计算也占一条流水。实测比值明显低于访存量之比 |
| <strong>融合的收益边界</strong> | 融合把计算集中到一个核函数里。当矢量流水越过搬运成为最慢的一条，再减少访存也不会更快——本实验的算子链已在边界上 |
| 中间张量 | 既非输入也非输出，须占用额外设备内存；v1 为 16 MiB，v3 与 v4 为 0 |
| Stream 的串行语义 | 同一 Stream 内任务按下发顺序执行，天然保证算子链的先后；下发本身是异步的，减少启动次数的收益远小于减少访存量的收益 |
| 核函数的启动开销 | 官方称为头开销，包含核启动、核取址 TLB MISS、同地址访问与变量资源初始化，并随核数增加而增加；空 Kernel 的耗时即其量级 |
| 就地复用 | 基础算术 API 允许源与目的重叠；把中间量写在输出缓冲上即可取消 <code>TBuf</code> |
| 高阶 API 的限制 | 不支持源操作数与目的操作数地址重叠，也不支持 <code>sharedTmpBuffer</code> 与源、目的地址重叠，因此无法参与就地复用 |
| 精度由最弱的一环决定 | 四个版本的最大相对误差完全相同，与实验四中 <code>Reciprocal</code> 的结果一致；融合不引入额外的精度损失 |
| 卸载的判断单位 | 融合不减少主机与设备之间的往返；是否值得卸载，应当以整段计算图为单位判断，而不是单个算子 |
| 融合的代价 | 片上压力上升、缓冲占用时间变长、计算集中、粒度变粗、可选实现受限 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合的收益来源</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不改变本实验这条链的计算量，减少中间结果在 Global Memory 上的往返；官方将其列为高优先级优化项「通过 Unified Buffer 融合实现连续 vector 计算」</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">访存量核算</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">逐核函数累加读写字节数；本链为 $28N \to 20N \to 12N$</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>读与写的字节数应当相加</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">MTE2 与 MTE3 各自独立发起搬运，却共用同一条 Global Memory 通路。官方给出：两者同时读写时，搬运流水的耗时按（MTE2 搬运量 + MTE3 搬运量）÷ Global Memory 带宽核算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>节拍由最慢的一条流水决定</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运与计算可以在时间上重叠；一个分块的时间取搬运总量所需的时间与矢量指令所需的时间之中较大的一个</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>访存量之比是宽松的上界</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">它假定耗时与访存量成正比，忽略了计算也占一条流水。实测比值明显低于访存量之比</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>融合的收益边界</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合把计算集中到一个核函数里。当矢量流水越过搬运成为最慢的一条，再减少访存也不会更快——本实验的算子链已在边界上</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中间张量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">既非输入也非输出，须占用额外设备内存；v1 为 16 MiB，v3 与 v4 为 0</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 的串行语义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同一 Stream 内任务按下发顺序执行，天然保证算子链的先后；下发本身是异步的，减少启动次数的收益远小于减少访存量的收益</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核函数的启动开销</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方称为头开销，包含核启动、核取址 TLB MISS、同地址访问与变量资源初始化，并随核数增加而增加；空 Kernel 的耗时即其量级</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">就地复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基础算术 API 允许源与目的重叠；把中间量写在输出缓冲上即可取消 <code>TBuf</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">高阶 API 的限制</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不支持源操作数与目的操作数地址重叠，也不支持 <code>sharedTmpBuffer</code> 与源、目的地址重叠，因此无法参与就地复用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">精度由最弱的一环决定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四个版本的最大相对误差完全相同，与实验四中 <code>Reciprocal</code> 的结果一致；融合不引入额外的精度损失</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">卸载的判断单位</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合不减少主机与设备之间的往返；是否值得卸载，应当以整段计算图为单位判断，而不是单个算子</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合的代价</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上压力上升、缓冲占用时间变长、计算集中、粒度变粗、可选实现受限</td>
</tr>
</tbody>
</table>

### 两条贯穿本章的原则

> **在异构算子开发中，性能问题首先是数据摆放问题，其次才是计算问题。**

> **每一项优化都有它的边界。** 判断一项优化还能不能继续做下去，要看瓶颈是否已经转移到别处。本实验把访存量之比与实测之比之间的差距归到了一个可以指出来的原因上：融合把计算集中到一个核函数，矢量流水因此越过搬运成为最慢的一条。

### 与后续实验的衔接

➡️ **后续内容：实验六 · 矩阵乘法 MatMul**。本实验在最后一步走到了收益边界：计算越过搬运成为瓶颈，继续减少访存不再有用。矩阵乘法处在这条边界的另一侧——它的算术强度随分块尺寸增长，可以远离受访存限制的区间。届时优化的方向随之改变，从减少访存转向充分利用计算单元，并第一次用到 Cube 单元。
